In [ ]:
class MyStack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def __len__(self):
        return len(self._items)

In [ ]:
class MySequence:
    def __getitem__(self, index):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __contains__(self, value):
        for v in self:
            if v == value:
                return True
        return False

In [ ]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]

f = Foo()
print(f[1])
print(20 in f)
print(15 in f)
for i in f:
    print(i)

In [ ]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

In [ ]:
from random import shuffle

deck = FrenchDeck()
shuffle(deck)

In [ ]:
def set_card(deck, position, card):
    deck._cards[position] = card

FrenchDeck.__setitem__ = set_card

from random import shuffle
shuffle(deck)
print(deck[:5])

In [ ]:
import collections
from collections.abc import MutableSequence

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

In [ ]:
import abc

class Drawable(abc.ABC):
    @abc.abstractmethod
    def draw(self):
        pass

class ExternalPainter:
    def draw(self):
        print("ExternalPainter draws something")

Drawable.register(ExternalPainter)

ep = ExternalPainter()
print(isinstance(ep, Drawable))
print(issubclass(ExternalPainter, Drawable))

In [ ]:
from collections.abc import Sized

class MyCollection:
    def __init__(self):
        self._items = [1, 2, 3]

    def __len__(self):
        return len(self._items)

Sized.register(MyCollection)

mc = MyCollection()
print(isinstance(mc, Sized))

In [ ]:
import abc

class Tombola(abc.ABC):
    @abc.abstractmethod
    def load(self, iterable):
        pass

    @abc.abstractmethod
    def pick(self):
        pass

    def loaded(self):
        return bool(self.inspect())

    def inspect(self):
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))

In [ ]:
import random

class BingoCage(Tombola):
    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()

In [ ]:
import random

class LotteryBlower(Tombola):
    def __init__(self, iterable):
        self._balls = list(iterable)

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty LotteryBlower')
        self._balls[position], self._balls[-1] = self._balls[-1], self._balls[position]
        return self._balls.pop()

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self._balls))

In [ ]:
from collections.abc import MutableSequence
import random

class TomboList(MutableSequence, Tombola):
    def __init__(self, iterable):
        self._items = list(iterable)

    def __getitem__(self, index):
        return self._items[index]

    def __setitem__(self, index, value):
        self._items[index] = value

    def __delitem__(self, index):
        del self._items[index]

    def __len__(self):
        return len(self._items)

    def insert(self, index, value):
        self._items.insert(index, value)

    def load(self, iterable):
        self._items.extend(iterable)

    def pick(self):
        if self._items:
            position = random.randrange(len(self._items))
            return self._items.pop(position)
        else:
            raise LookupError('pick from empty TomboList')

In [ ]:
for cls in [BingoCage, LotteryBlower, TomboList]:
    t = cls(range(1, 11))
    print(f'{cls.__name__}:')
    print(f'  loaded: {t.loaded()}')
    print(f'  inspect: {t.inspect()}')
    print(f'  pick: {t.pick()}')
    print(f'  inspect after pick: {t.inspect()}')